# Load the data; you must run this section 3 times



Install packages


In [ ]:
!pip install awscli
!pip install h5py
!pip install -q transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: rsa
    Found existing installation: rsa 4.9.1
    Uninstalling rsa-4.9.1:
      Successfully uninstalled rsa-4.9.1
  Attempting uninstall: docutils
    Found existing installation: docutils 0.21.2
    Uninstalling docutils-0.21.2:
      Successfully uninstalled docutils-0.21.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


Import packages

In [ ]:
import os
import torch
import h5py
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from transformers import CLIPTokenizer, CLIPTextModel
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Download Subject X processed fMRI responses from OpenNeuro

you must change sub_id: 01 to 02, 03

In [ ]:
sub_id = 3

!aws s3 sync --no-sign-request \
s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-0{sub_id} \
/content/sub-0{sub_id}_ICA

download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_stimulus-metadata.json to sub-03_ICA/voxel-metadata/sub-03_task-things_stimulus-metadata.json
download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_voxel-wise-responses.json to sub-03_ICA/voxel-metadata/sub-03_task-things_voxel-wise-responses.json
download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_voxel-metadata.json to sub-03_ICA/voxel-metadata/sub-03_task-things_voxel-metadata.json
download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_stimulus-metadata.tsv to sub-03_ICA/voxel-metadata/sub-03_task-things_stimulus-metadata.tsv
download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_voxel-metadata.tsv to sub-03_ICA/voxel-metadata/sub-03_task-things_voxel-metadata.tsv
download: s3://openneuro.org/ds0041

Define filepaths

In [ ]:
data_root = f"/content/sub-0{sub_id}_ICA"

stim_meta_path = f"{data_root}/voxel-metadata/sub-0{sub_id}_task-things_stimulus-metadata.tsv"
h5_path = f"{data_root}/voxel-metadata/sub-0{sub_id}_task-things_voxel-wise-responses.h5"

Load stimulus data
This table links each trial to the object concept shown

In [ ]:
stim_meta = pd.read_csv(stim_meta_path, sep=",")

Load voxel-wise neural responses
rows are stimulus trials, columns are voxels

In [ ]:
with h5py.File(h5_path, "r") as f:
    Y_all_trials = f["ResponseData"]["block0_values"][()]

Compute average neural response per concept

In [ ]:
concepts = stim_meta["concept"].unique()
n_voxels = Y_all_trials.shape[1]

Y_objects = np.zeros((len(concepts), n_voxels), dtype=np.float32)
concept_mapping = []

for i, concept in enumerate(concepts):

    idx = stim_meta[stim_meta["concept"] == concept].index

    Y_objects[i] = Y_all_trials[idx].mean(axis=0)

    concept_mapping.append({
        "concept": concept,
        "stimulus_ids": list(stim_meta.loc[idx, "stimulus"]),
        "n_trials": len(idx)
    })

Save processed outputs

In [ ]:
np.save(f"{data_root}/Y_sub0{sub_id}_visual.npy", Y_objects)

pd.DataFrame(concept_mapping).to_csv(
    f"{data_root}/concepts_sub0{sub_id}.csv",
    index=False
)

print(f"Sub-0{sub_id} done: {Y_objects.shape[0]} concepts × {Y_objects.shape[1]} voxels")

Sub-03 done: 720 concepts × 9840 voxels


clear RAM

In [ ]:
del Y_all_trials, Y_objects, stim_meta

After running this clear RAM line then you can go back to the top and change the subject number. If you do not clear RAM between subjects the code may not run

# All 3 Subject Data should be downloaded

Quick look at the data

In [ ]:
# check each participant folder
subs = [1,2,3]

for sub in subs:

    root = f"/content/sub-0{sub}_ICA"

    print(f"\n===== SUBJECT {sub} =====")

    # load neural matrix
    Y = np.load(f"{root}/Y_sub0{sub}_visual.npy")

    # load concept metadata
    concepts = pd.read_csv(f"{root}/concepts_sub0{sub}.csv")

    print("Neural matrix shape:", Y.shape)
    print("First 5 concepts:")
    print(concepts.head())

    print("Example neural vector (first concept, first 10 voxels):")
    print(Y[0][:10])

    print("Number of concepts:", len(concepts))


===== SUBJECT 1 =====
Neural matrix shape: (720, 9840)
First 5 concepts:
      concept                                       stimulus_ids  n_trials
0         dog  ['dog_12s.jpg', 'dog_04s.jpg', 'dog_01b.jpg', ...        12
1       mango  ['mango_12s.jpg', 'mango_13s.jpg', 'mango_13s....        24
2     spatula  ['spatula_12s.jpg', 'spatula_04s.jpg', 'spatul...        12
3  candelabra  ['candelabra_14s.jpg', 'candelabra_12s.jpg', '...        24
4       panda  ['panda_12s.jpg', 'panda_04s.jpg', 'panda_01b....        12
Example neural vector (first concept, first 10 voxels):
[-0.00996814  0.00019548  0.01672786 -0.00846357 -0.00021558  0.01358145
  0.02329377  0.02388763  0.02475897  0.06129166]
Number of concepts: 720

===== SUBJECT 2 =====
Neural matrix shape: (720, 9840)
First 5 concepts:
   concept                                       stimulus_ids  n_trials
0     bird  ['bird_06s.jpg', 'bird_01b.jpg', 'bird_08s.jpg...        12
1  compass  ['compass_06n.jpg', 'compass_01b.jpg', 'com

Check our concepts before computing embeddings

In [ ]:
concepts = pd.read_csv("/content/sub-01_ICA/concepts_sub01.csv")["concept"].tolist()

print(len(concepts))
print(concepts[:30])

720
['dog', 'mango', 'spatula', 'candelabra', 'panda', 'pacifier', 'crayfish', 'shirt', 'chicken_wire', 'cow', 'bib', 'altar', 'beaker', 'knife', 'wheelchair', 'videocassette', 'lizard', 'canoe', 'cookie', 'ukulele', 'stove1', 'hammock', 'dish', 'rat', 'blind', 'treadmill', 'gondola', 'pear', 'drum', 'chess_piece']


Load list of all 720 concepts

In [ ]:
concepts = pd.read_csv("/content/sub-01_ICA/concepts_sub01.csv")["concept"].tolist()

print("Number of concepts:", len(concepts))
print("Example concepts:", concepts[:10])

Number of concepts: 720
Example concepts: ['dog', 'mango', 'spatula', 'candelabra', 'panda', 'pacifier', 'crayfish', 'shirt', 'chicken_wire', 'cow']


Load CLIP test encoder

In [ ]:
model_name = "openai/clip-vit-base-patch32"

tokenizer = CLIPTokenizer.from_pretrained(model_name)
model = CLIPTextModel.from_pretrained(model_name)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
visual_projection.weight                                       | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.

CLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

Compute semantic embeddings

In [ ]:
embeddings = []

with torch.no_grad():

    for concept in concepts:

        inputs = tokenizer(concept, return_tensors="pt")
        outputs = model(**inputs)

        # Mean pooling across token embeddings
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

        embeddings.append(embedding)

X = np.vstack(embeddings)

print("Embedding matrix shape:", X.shape)

Embedding matrix shape: (720, 512)


Save final X and Y matrices

In [ ]:
np.save("/content/X_semantic_embeddings_clip.npy", X)

Compare the concepts across participants

In [ ]:
# List of subjects
subs = [1, 2, 3]

# Load concept sets for each subject
concept_sets = []
for sub in subs:
    meta = pd.read_csv(f"/content/sub-0{sub}_ICA/concepts_sub0{sub}.csv")
    concept_sets.append(set(meta["concept"]))

# Compute common concepts across all subjects
common_concepts = sorted(set.intersection(*concept_sets))

print("Number of common concepts:", len(common_concepts))
print("Example common concepts:", common_concepts[:10])

Number of common concepts: 720
Example common concepts: ['acorn', 'airbag', 'aircraft_carrier', 'airplane', 'alligator', 'aloe', 'altar', 'aluminum_foil', 'anchor', 'anklet']


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# --- Set save location ---
save_path = Path("/content/my_drive/MyDrive/things_fmri_project")
save_path.mkdir(parents=True, exist_ok=True)

# --- Save semantic embeddings ---
np.save(save_path / "X_semantic_embeddings_clip.npy", X)

# --- Save voxel matrices, trial counts, and concept metadata ---
for sub in subs:
    root = f"/content/sub-0{sub}_ICA"

    # Load previously saved Y matrix and concept metadata
    Y = np.load(f"{root}/Y_sub0{sub}_visual.npy")
    concepts_meta = pd.read_csv(f"{root}/concepts_sub0{sub}.csv")

    # Save Y matrix
    np.save(save_path / f"Y_sub{sub}_visual.npy", Y)

    # Save trial counts per concept
    trial_counts = concepts_meta[['concept', 'n_trials']]
    trial_counts.to_csv(save_path / f"Y_sub{sub}_trial_counts.csv", index=False)

    # Save full concept metadata
    concepts_meta.to_csv(save_path / f"concepts_sub{sub}.csv", index=False)

# --- Save common concept list ---
pd.Series(common_concepts).to_csv(save_path / "concept_list.csv", index=False)

print("All data saved. Files in project folder:")
print(sorted([f.name for f in save_path.iterdir()]))

All data saved. Files in project folder:
['X_semantic_embeddings_clip.npy', 'Y_sub1_trial_counts.csv', 'Y_sub1_visual.npy', 'Y_sub2_trial_counts.csv', 'Y_sub2_visual.npy', 'Y_sub3_trial_counts.csv', 'Y_sub3_visual.npy', 'concept_list.csv', 'concepts_sub1.csv', 'concepts_sub2.csv', 'concepts_sub3.csv']


Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/my_drive')

Mounted at /content/my_drive
